# Preparing a Dataset for Model Training

This set of scripts is designed to prepare a structured dataset (images and masks) from various geographical 'sites' for subsequent training, validation, and testing of a segmentation model.

### Key Steps:

1.  **Configuration of Paths and Parameters:**
    *   `PROJECT_ROOT`: The root path of your project in Google Drive.
    *   `INPUT_PATCHES_FOLDER`: The name of the folder where raw patches are stored (e.g., `patches_PS_14_bands_excluding_blue`).
    *   `OUTPUT_PATCHES_FOLDER`: The name of the folder for saving the prepared dataset.
    *   `MAX_TRAIN_PATCHES_PER_OLD_SITE`, `MAX_TRAIN_PATCHES_PER_NEW_SITE`, `MAX_VAL_PATCHES_PER_SITE`, `MAX_TEST_PATCHES_PER_SITE`: Define the maximum number of patches selected for each site in the corresponding set (training, validation, test). This helps control the size and balance of the dataset.

2.  **Directory Initialization:**
    *   The code clears (if it exists) and creates a new directory structure for storing the final dataset: `DATASET_FOLDER/train/images`, `DATASET_FOLDER/train/labels`, `DATASET_FOLDER/val/images`, `DATASET_FOLDER/val/labels`, `DATASET_FOLDER/test/images`, `DATASET_FOLDER/test/labels`.

3.  **Patch Collection and Sampling:**
    *   The `collect_split_samples` function gathers image-mask pairs from the specified path and split type (`'train'`, `'val'`, `'test'`).
    *   **Distinction between `OLD_SITES` and `NEW_SITES`:**
        *   `NEW_SITES`: A list of sites considered as new data for training. For these, a larger number of patches is typically used (`MAX_TRAIN_PATCHES_PER_NEW_SITE`).
        *   `OLD_SITES`: A list of sites that have been used previously or represent known data. If `OLD_SITES` **contain data**, they can be included in the training set. This is particularly important for **fine-tuning**, where a model has already been trained on a base dataset (potentially including `OLD_SITES`) and is now being adapted to new conditions or sites (`NEW_SITES`). Including `OLD_SITES` in this scenario helps the model not to "forget" previously acquired knowledge. If `OLD_SITES` **is empty**, then model training proceeds only on data from `NEW_SITES`, which is characteristic of training "from scratch" on a new dataset.
    *   For each site in `OLD_SITES` and `NEW_SITES`, a random sample of patches (images and masks) is selected for the training, validation, and test sets, according to the specified `MAX_...` parameters.

4.  **File Copying:**
    *   Patches from `train_samples`, `val_samples`, and `test_samples` are copied to the respective directories within `DATASET_FOLDER`.
    *   The use of `ThreadPoolExecutor` and `tqdm` ensures efficient multi-threaded file copying with a progress indicator.

5.  **Metadata Generation:**
    *   Finally, a `metadata.json` file is created in the root of `DATASET_FOLDER`. This file contains important information about the dataset, such as the number of classes, patch size, number of channels, and detailed patch counts for each site and each split type (train/val/test). This is useful for subsequent use of the dataset in model training.

# ⚙️ Configuration (set the parameters)

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/U-Net_burned_areas_project"

INPUT_PATCHES_FOLDER = 'patches_PS_SENT_432_682' # Just the folder name #
OUTPUT_PATCHES_FOLDER = 'dataset_PS_SENT_432_682' # Just the folder name #

MAX_TRAIN_PATCHES_PER_OLD_SITE = 300
MAX_TRAIN_PATCHES_PER_NEW_SITE = 1000

MAX_VAL_PATCHES_PER_SITE = 400
MAX_TEST_PATCHES_PER_SITE = 200

OLD_SITES = [
    # "site-969-tropical-savanna",
    # "site-789-tropical-savanna", for example
    # "site-385-tropical-savanna",#
    # "site-413-tropical-savanna",#
]

NEW_SITES = [
    # "site-969-tropical-savanna",
    "site-432-tropical-savanna",
    # "site-789-tropical-savanna",
    # "site-969-tropical-savanna",
    "site-682-tropical-savanna",
]

#⏳ Creation of patches... (run all the cells)

In [ ]:
import random
from pathlib import Path
import shutil
import json

In [ ]:
from tqdm import tqdm

In [ ]:
PROJECT_ROOT = Path(PROJECT_ROOT)
DATASET_FOLDER = PROJECT_ROOT / 'data' / 'merged_patches_datasets' / OUTPUT_PATCHES_FOLDER

INPUT_PATCHES_ROOT = Path(f"{PROJECT_ROOT}/data/sites")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
TRAIN_IMG_DIR = DATASET_FOLDER / "train/images"
TRAIN_MASK_DIR = DATASET_FOLDER / "train/labels"

VAL_IMG_DIR = DATASET_FOLDER / "val/images"
VAL_MASK_DIR = DATASET_FOLDER / "val/labels"

In [ ]:
TEST_IMG_DIR = DATASET_FOLDER / "test/images"
TEST_MASK_DIR = DATASET_FOLDER / "test/labels"

In [ ]:
if DATASET_FOLDER.exists():
    shutil.rmtree(DATASET_FOLDER)

TRAIN_IMG_DIR.mkdir(parents=True)
TRAIN_MASK_DIR.mkdir(parents=True)

VAL_IMG_DIR.mkdir(parents=True)
VAL_MASK_DIR.mkdir(parents=True)

In [ ]:
TEST_IMG_DIR.mkdir(parents=True)
TEST_MASK_DIR.mkdir(parents=True)

In [ ]:
def collect_split_samples(site_path, split="train"):

    images = sorted(
        (site_path / split / "images").glob("*")
    )

    labels = sorted(
        (site_path / split / "labels").glob("*")
    )

    return list(zip(images, labels))

In [ ]:
train_samples = []
val_samples = []

In [ ]:
import random
from pathlib import Path
import shutil
import json

# Choose one site to read metadata from (e.g., the first new site)
example_site = NEW_SITES[0]
metadata_path = INPUT_PATCHES_ROOT / example_site / 'patches' / INPUT_PATCHES_FOLDER / "metadata.json"

if not metadata_path.exists():
    raise FileNotFoundError(f"Metadata file not found at: {metadata_path}")

with open(metadata_path, 'r') as f:
    site_metadata = json.load(f)

NUM_CLASSES = site_metadata.get('n_classes')
INPUT_SIZE = site_metadata.get('patch_size')
NUM_BANDS = site_metadata.get('n_bands_total')

# Extract band information
PS_BANDS = site_metadata.get('ps_bands')
S2_BANDS = site_metadata.get('s2_bands')
LS_BANDS = site_metadata.get('ls_bands')

print(f"NUM_CLASSES: {NUM_CLASSES}")
print(f"INPUT_SIZE: {INPUT_SIZE}")
print(f"NUM_BANDS: {NUM_BANDS}")
print(f"PS_BANDS: {PS_BANDS}")
print(f"S2_BANDS: {S2_BANDS}")
print(f"LS_BANDS: {LS_BANDS}")

if any(x is None for x in [
        NUM_CLASSES,
        INPUT_SIZE,
        NUM_BANDS,
        PS_BANDS,
        S2_BANDS,
        LS_BANDS
    ]):
    print("Warning: Some metadata values could not be loaded. Please check the metadata.json structure.")

NUM_CLASSES: 2
INPUT_SIZE: 128
NUM_BANDS: 30
PS_BANDS: [2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16]
S2_BANDS: [5, 6, 7, 11, 12]
LS_BANDS: []


In [ ]:
for site in NEW_SITES:

    site_path = INPUT_PATCHES_ROOT / site / 'patches' / INPUT_PATCHES_FOLDER

    samples = collect_split_samples(
        site_path,
        split="train"
    )

    sampled = random.sample(
        samples,
        min(MAX_TRAIN_PATCHES_PER_NEW_SITE, len(samples))
    )

    train_samples.extend(sampled)

In [ ]:
for site in OLD_SITES:

    site_path = INPUT_PATCHES_ROOT / site / 'patches' / INPUT_PATCHES_FOLDER

    samples = collect_split_samples(
        site_path,
        split="train"
    )

    sampled = random.sample(
        samples,
        min(MAX_TRAIN_PATCHES_PER_OLD_SITE, len(samples))
    )

    train_samples.extend(sampled)

In [ ]:
print(len(train_samples))

2000


In [ ]:
# MAX_VAL_PATCHES_PER_SITE = 4

for site in OLD_SITES + NEW_SITES:

    site_path = INPUT_PATCHES_ROOT / site / 'patches' / INPUT_PATCHES_FOLDER

    samples = collect_split_samples(
        site_path,
        split="val"
    )

    sampled = random.sample(
        samples,
        min(MAX_VAL_PATCHES_PER_SITE, len(samples))
    )

    val_samples.extend(sampled)

In [ ]:
print(f"Val samples: {len(val_samples)}")

Val samples: 662


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import shutil
from tqdm import tqdm

def copy_sample(args):
    img_path, lbl_path, img_dst, lbl_dst = args

    shutil.copy2(img_path, img_dst)
    shutil.copy2(lbl_path, lbl_dst)

tasks = []

for i, (img_path, lbl_path) in enumerate(train_samples):

    tasks.append((
        img_path,
        lbl_path,
        DATASET_FOLDER / "train/images" / f"{i}.npy",
        DATASET_FOLDER / "train/labels" / f"{i}.npy"
    ))

with ThreadPoolExecutor(max_workers=8) as executor:
    list(tqdm(
        executor.map(copy_sample, tasks),
        total=len(tasks)
    ))

100%|██████████| 2000/2000 [05:57<00:00,  5.59it/s]


In [ ]:
# Test patches

test_samples = []

for site in OLD_SITES + NEW_SITES:

    site_path = INPUT_PATCHES_ROOT / site / 'patches' / INPUT_PATCHES_FOLDER

    samples = collect_split_samples(
        site_path,
        split="test"
    )

    sampled = random.sample(
        samples,
        min(MAX_TEST_PATCHES_PER_SITE, len(samples))
    )

    test_samples.extend(sampled)

In [ ]:
print(len(test_samples))

400


In [ ]:
tasks = []

for i, (img_path, lbl_path) in enumerate(val_samples):

    tasks.append((
        img_path,
        lbl_path,
        DATASET_FOLDER / "val/images" / f"{i}.npy",
        DATASET_FOLDER / "val/labels" / f"{i}.npy"
    ))

with ThreadPoolExecutor(max_workers=8) as executor:
    list(tqdm(
        executor.map(copy_sample, tasks),
        total=len(tasks)
    ))

100%|██████████| 662/662 [01:12<00:00,  9.13it/s]


In [ ]:
tasks = []

for i, (img_path, lbl_path) in enumerate(test_samples):

    tasks.append((
        img_path,
        lbl_path,
        DATASET_FOLDER / "test/images" / f"{i}.npy",
        DATASET_FOLDER / "test/labels" / f"{i}.npy"
    ))

with ThreadPoolExecutor(max_workers=8) as executor:
    list(tqdm(
        executor.map(copy_sample, tasks),
        total=len(tasks)
    ))

100%|██████████| 400/400 [00:41<00:00,  9.76it/s]


In [ ]:
import json

# Calculate per-site counts
train_site_counts = {}
for img_path, _ in train_samples:
    # Corrected path traversal to get the actual site name
    site_name = img_path.parent.parent.parent.parent.parent.name
    train_site_counts[site_name] = train_site_counts.get(site_name, 0) + 1

val_site_counts = {}
for img_path, _ in val_samples:
    # Corrected path traversal to get the actual site name
    site_name = img_path.parent.parent.parent.parent.parent.name
    val_site_counts[site_name] = val_site_counts.get(site_name, 0) + 1

test_site_counts = {}
for img_path, _ in test_samples:
    # Corrected path traversal to get the actual site name
    site_name = img_path.parent.parent.parent.parent.parent.name
    test_site_counts[site_name] = test_site_counts.get(site_name, 0) + 1

detailed_site_counts = {}
all_sites = sorted(list(set(OLD_SITES + NEW_SITES)))
for site in all_sites:
    detailed_site_counts[site] = {
        "train": train_site_counts.get(site, 0),
        "val": val_site_counts.get(site, 0),
        "test": test_site_counts.get(site, 0)
    }


metadata = {
    "sites": NEW_SITES + OLD_SITES,
    "n_classes": NUM_CLASSES, # Assuming NUM_CLASSES is defined elsewhere
    "patch_size": INPUT_SIZE,
    "n_bands_total": NUM_BANDS,
    "ps_bands": PS_BANDS,
    "s2_bands": S2_BANDS,
    "ls_bands": LS_BANDS,
    "counts": {
        "train": len(train_samples),
        "val": len(val_samples),
        "test": len(test_samples)
    },
    "site_counts": detailed_site_counts
}

with open(DATASET_FOLDER / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

In [ ]:
# Verify Output Folder Contents
print(f"Checking contents of the output folder: {DATASET_FOLDER}")

if not DATASET_FOLDER.exists():
    print("Error: Output DATASET_FOLDER does not exist. There might be an issue with directory creation or the path itself.")
else:
    print(f"DATASET_FOLDER exists. Contents:\n")
    for split_dir in ["train", "val", "test"]:
        split_path = DATASET_FOLDER / split_dir
        if split_path.exists():
            print(f"  - {split_path.name}/")
            for sub_dir_name in ["images", "labels"]:
                sub_dir_path = split_path / sub_dir_name
                if sub_dir_path.exists():
                    print(f"    - {sub_dir_name}/")
                    contents = list(sub_dir_path.iterdir())
                    if contents:
                        print(f"      (First 5 files in {sub_dir_name}):")
                        for item in contents[:5]:
                            print(f"        {item.name}")
                    else:
                        print(f"      {sub_dir_name} directory is empty.")
                else:
                    print(f"    - {sub_dir_name}/ (does not exist)")
        else:
            print(f"  - {split_path.name}/ (does not exist)")


Checking contents of the output folder: /content/drive/MyDrive/Geoinformatics_Project/NewStructure/data/merged_patches_datasets/dataset_PS_SENT_432_682
DATASET_FOLDER exists. Contents:

  - train/
    - images/
      (First 5 files in images):
        7.npy
        4.npy
        5.npy
        6.npy
        3.npy
    - labels/
      (First 5 files in labels):
        7.npy
        3.npy
        2.npy
        4.npy
        1.npy
  - val/
    - images/
      (First 5 files in images):
        3.npy
        2.npy
        0.npy
        1.npy
        6.npy
    - labels/
      (First 5 files in labels):
        1.npy
        3.npy
        6.npy
        7.npy
        0.npy
  - test/
    - images/
      (First 5 files in images):
        0.npy
        2.npy
        1.npy
        3.npy
        5.npy
    - labels/
      (First 5 files in labels):
        2.npy
        6.npy
        3.npy
        0.npy
        1.npy


##### Contributors:
- **Evgenii Miasnikov**: evgenii.miasnikov@mail.polimi.it
- **Ayman Mutasim Alfadul Abdelgadir**: aymanmutasim@mail.polimi.it

```
   Copyright 2026 Evgenii Miasnikov, Ayman Mutasim Alfadul Abdelgadir

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       https://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.
```